In [1]:
from abc import ABC, abstractmethod

In [2]:
class Runnable(ABC):

    @abstractmethod
    def invoke(self, input_data):
        pass

In [3]:
import random

class NakliLLM(Runnable): 

    def __init__(self):
        print('LLM start')

    def invoke(self, prompt):
        response_list = [
            'Mumbai is Capital of Maharashtra',
            'Hero ISL is a Foodball leage',
            'AI stand for Artificial Intelligence'
        ]

        return {"response": random.choice(response_list)}

    def predict(self, prompt):
        response_list = [
            'Mumbai is Capital of Maharashtra',
            'Hero ISL is a Foodball leage',
            'AI stand for Artificial Intelligence'
        ]

        return {"response": random.choice(response_list)}

In [5]:
class NakliPromptTemplate(Runnable):

    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def invoke(self, input_dict):
        if isinstance(input_dict, dict):
            return self.template.format(**input_dict)
        elif isinstance(input_dict, str) and self.input_variables:
            return self.template.format(**{self.input_variables[0]: input_dict})
        return self.template

    def format(self, input_dict):
        return self.invoke(input_dict)

In [11]:
class NakliStrOutParser(Runnable):
    def __init__(self):
        pass
    def invoke(self, input_data):
        if isinstance(input_data, dict) and 'response' in input_data:
            return input_data['response']
        return str(input_data)

In [6]:
class RunnableConnector(Runnable):

    def __init__(self, runnable_list):
        self.runnable_list = list(runnable_list)

    def invoke(self, input_data):
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)
        return input_data

In [7]:
template = NakliPromptTemplate(
    template = 'Write a {length} poem about {topic}',
    input_variables = ['length', 'topic']
) 

In [8]:
llm = NakliLLM()

LLM start


In [12]:
parser = NakliStrOutParser()

In [14]:
chain = RunnableConnector([template, llm])

In [17]:
chain.invoke({'length':'long', 'topic':'india'})

{'response': 'Mumbai is Capital of Maharashtra'}

In [25]:
template1 = NakliPromptTemplate(
    template = 'Write a joke about {topic}',
    input_variables = ['topic']
)

In [31]:
template2 = NakliPromptTemplate(
    template = 'Explain the following joke {response}',
    input_variables = ['response']
)

In [26]:
llm = NakliLLM()

LLM start


In [27]:
parser = NakliStrOutParser()

In [28]:
chain1 = RunnableConnector([template1, llm])


In [29]:
chain1.invoke({'topic':'AI'})

{'response': 'Mumbai is Capital of Maharashtra'}

In [32]:
chain2 = RunnableConnector([template2, llm, parser])


In [35]:
final_chain = RunnableConnector([chain1, chain2])

In [38]:
final_chain.invoke({'topic':'cricket'})

TypeError: str.format() argument after ** must be a mapping, not str